# 03 — The transformation pipeline

The *Transform* step: going from heterogeneous raw publications to a clean, typed dataset
that conforms to the schema. The pipeline is laid out in three movements — **read**,
**process**, **export** — and every single transformation is a small, testable function.

In [1]:
from dotenv import load_dotenv

from multimodal_etl.logging_setup import setup_logging
from multimodal_etl.utils.paths import ROOT_DIR

# No `sys.path` insert: the project is installed in the environment `uv run` provides, and a
# notebook that patches the path hides which interpreter it is actually running against.
load_dotenv(ROOT_DIR / ".env")
setup_logging()

## 1. The single-purpose functions

Each does one thing, which makes them readable and testable separately
(`tests/test_transform.py`).

In [2]:
from multimodal_etl.transform import (
    clean_text,
    extract_domain,
    normalise_date,
    normalise_label,
)

print(clean_text("<p>A   summary &amp; its <b>HTML</b></p>"))
print(extract_domain("https://www.bbc.co.uk/news/article-123"))
print(normalise_date("Mon, 29 Jun 2026 10:00:00 GMT"))
print(normalise_date("1767225600"))
print(normalise_label("FAKE"), normalise_label("Real"), normalise_label(""))

A summary & its HTML
bbc.co.uk
2026-06-29T10:00:00+00:00
2026-01-01T00:00:00+00:00
fake real None


Dates deserve a word: every source has its own format — RFC 822 for RSS, ISO for the API,
a Unix timestamp for Fakeddit. Without normalising, the freshness indicator would simply be
impossible to compute.

## 2. The rule that defines the dataset

`validate_image` does not look at the URL: it checks that **the file is on disk**. The cell
below shows it refusing a path that leads nowhere, and accepting one that leads to a real
file.

In [3]:
from multimodal_etl.transform import validate_image

print(validate_image("data/raw/images/does-not-exist.jpg"))
print(validate_image(""))

False
False


## 3. Read → process → export

In [4]:
from multimodal_etl.config import RAW_DIR, TransformConfig
from multimodal_etl.transform import export_dataset, process, read_raw

config = TransformConfig()
latest_raw = sorted(RAW_DIR.glob("raw_publications_*.json"))[-1]
raw = read_raw(latest_raw)
print(f"{len(raw)} raw publications read")

2026-09-15 13:52:49 | INFO    | multimodal_etl.transform | Transform: reading raw file G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\raw\raw_publications_20260915_115245.json


2026-09-15 13:52:49 | INFO    | multimodal_etl.transform | Transform: 171 raw publications read


171 raw publications read


In [5]:
df, stats = process(raw, config)
stats

2026-09-15 13:52:49 | INFO    | multimodal_etl.transform | Transform: 131/171 publications valid after cleaning


2026-09-15 13:52:49 | INFO    | multimodal_etl.transform | Transform: 0 duplicates removed


{'raw_total': 171,
 'valid_total': 131,
 'rejected': 40,
 'duplicates': 0,
 'with_image': 131,
 'labelled': 32,
 'native_images': 123,
 'open_graph_images': 8}

The rejection rate is high, and that is normal: almost every dropped publication goes for
a single reason — no usable image. Let us check that.

In [6]:
reasons = {"empty title": 0, "text too short": 0, "no image": 0, "kept": 0}
for publication in raw:
    if not clean_text(str(publication.get("title", ""))):
        reasons["empty title"] += 1
    elif len(clean_text(str(publication.get("text", "")))) < config.min_text_length:
        reasons["text too short"] += 1
    elif not validate_image(str(publication.get("image_path", ""))):
        reasons["no image"] += 1
    else:
        reasons["kept"] += 1
reasons

{'empty title': 0, 'text too short': 4, 'no image': 36, 'kept': 131}

## 4. The dataset produced

The columns come straight from `multimodal_etl.schema`, the single source of truth shared
by the code, the diagram and the documentation.

In [7]:
df[["source", "access_method", "title", "image_source", "has_image", "label"]].head(8)

,source,access_method,title,image_source,has_image,label
0,rss:the_guardian,rss_feed,‘I believed the hype’: the white South African...,native,True,NaN
1,rss:the_guardian,rss_feed,"Ebola outbreak in DRC has peaked, say authorit...",native,True,NaN
2,rss:the_guardian,rss_feed,The London hospital funded by donors who inves...,native,True,NaN
3,rss:the_guardian,rss_feed,Africa’s richest man aiming to make $23bn from...,native,True,NaN
4,rss:the_guardian,rss_feed,Burial of King Oyo takes place in Uganda as ne...,native,True,NaN
5,rss:the_guardian,rss_feed,Panama canal traffic to be cut again as drough...,native,True,NaN
6,rss:the_guardian,rss_feed,‘A coin flip’: on the campaign trail as Brazil...,native,True,NaN
7,rss:the_guardian,rss_feed,Canada offers air defense support for Ukraine ...,native,True,NaN


In [8]:
print("By access method:")
print(df["access_method"].value_counts().to_string())
print()
print("Where the images came from:")
print(df["image_source"].value_counts().to_string())

By access method:
access_method
rss_feed           99
kaggle_download    24
github_download     8

Where the images came from:
image_source
native        123
open_graph      8


## 5. Export

The format chosen is **Parquet**: columnar, typed, compact. By this stage the schema is
fixed and the file is meant for analytical reads. The statistics are written next to it,
beside it: the cell below writes both, and the dashboard reads them together.

In [9]:
dataset_path = export_dataset(df, config, stats)
print("Dataset:   ", dataset_path.name)
print("Statistics:", dataset_path.with_name(dataset_path.stem + "_stats.json").name)

2026-09-15 13:52:53 | INFO    | multimodal_etl.transform | Transform: dataset of 131 rows exported to G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\processed\publications_20260915_115250.parquet


2026-09-15 13:52:53 | INFO    | multimodal_etl.transform | Transform: statistics written to G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\processed\publications_20260915_115250_stats.json


Dataset:    publications_20260915_115250.parquet
Statistics: publications_20260915_115250_stats.json
